# Group B: Kernel Functions

Background: It is easy to take a lot of the math that our computers do for granted. There are actual functions that people programmed that drive all of the math. However, most functions that you have worked with were implemented on CPU. Numpy arrays are computed by your cpu. In recent times, libraries like Pytorch can run on your GPU. The GPU computes things in parallel, turning sequential operations into things that can be computed simultaneously.

In some cases, you can turn things that would originally take O(n) time in to things that take O(1) time.

Your job is to write these math functions in **Triton**, the python library for GPU programming.

Triton Documentation: https://triton-lang.org/main/index.html

Deliverables:

* Vector Addition
* Mean
* GeMM
* Relu
* Fused Softmax
* Batch Norm
* Max Pooling
* Global Average Pooling


Possibly Helpful Youtube Tutorial: The dude has a playlist. I watched some of his videos but he's kind of annoying...

https://www.youtube.com/watch?v=FtmnriHLbAg&list=PL_NMDNzkCbLR69QafQUa6yTx-T-VPIoaZ


# Kernel Functions

The following are series of basic tensor operations optimized with Triton. Triton comes with a lot of conventions which are helpful in interpreting the following code: they're listed below
- CPU/GPU code separation: most major operations are split into a cpu function and a gpu function (for example, vector addition is split into `add_kernel` and `vector_add`, the GPU and CPU workloads, respectively, with the CPU functino typically calling the GPU function)
- The kernel is named `[operation]_kernel`, with the wrapper (CPU function) named semantically (e.g. `vector_add`)
- `BLOCK_SIZE`=1024 by default and must be a power of 2 (declared `tl.constexpr)
- Always pass `n` (number of elements) explicitly into the kernel
- Allocate the output tensor with `torch.empty_like` in the wrapper (CPU function)
- Always mask loads and stores with offs < n
- Wrapper should validate inputs (e.g. .is_cuda())

Comments included liberally

In [ ]:
!pip install torch
!pip install triton

In [ ]:
import torch
import triton
import triton.language as tl

---
### Vector Addition

Cell 1: GPU kernel to add two tiles

Cell 2: CPU wrapper to add two arrays (torch tensors)

Cell 3: Usage demo

In [ ]:
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr): # GPU function
    """
    x_ptr: pointer (integer that points to the memory address of the start) for the first arr
    y_ptr: pointer for 2nd arr
    out_ptr: pointer for where to store the result
    # n is the number of elements in the array 
    # BLOCK_SIZE: number of elements each program processes (likely don't have to edit)
    """
    pid = tl.program_id(0) # identify tile
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE) # global indices for the tile
    mask = offs < n # prevent going out of bounds past last tile
    tl.store(out_ptr + offs,
             tl.load(x_ptr + offs, mask=mask) + tl.load(y_ptr + offs, mask=mask), # load both tiles, add elementwise
             mask=mask)

In [ ]:
def vector_add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    assert x.is_cuda and y.is_cuda, "inputs must be CUDA tensors"
    assert x.shape == y.shape
    out = torch.empty_like(x) # allocate space for the output
    n = x.numel() # len(x)
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(n, BLOCK_SIZE),) # number of programs to launch (ceil(n / BLOCK_SIZE))
    add_kernel[grid](x, y, out, n, BLOCK_SIZE=BLOCK_SIZE) # add_kernel[grid] returns a callable s.t. each tile runs through the addition
    return out

In [ ]:
n = 2 ** 20 # 1M elements
x = torch.rand(n, device='cuda', dtype=torch.float32)
y = torch.rand(n, device='cuda', dtype=torch.float32)
out = vector_add(x, y)

# verify
assert torch.allclose(out, x + y)
print("max error:", (out - (x + y)).abs().max().item())

---
### ReLU

ReLU applies $f(x) = \max(0, x)$ elementwise

Cell 1: GPU kernel to apply ReLU over a tile

Cell 2: CPU wrapper

Cell 3: Usage demo

In [ ]:
@triton.jit
def relu_kernel(x_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0) # identify tile
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE) # global indices for this tile
    mask = offs < n # prevent out of bounds
    x = tl.load(x_ptr + offs, mask=mask, other=0.0) # load tile; OOB slots get 0.0 (safe for relu)
    tl.store(out_ptr + offs, tl.where(x >= 0, x, 0.0), mask=mask) # ReLU (mask blocks OOB writes)

In [ ]:
def relu(x: torch.Tensor) -> torch.Tensor: # this is more or less same as before so won't add comments
    assert x.is_cuda
    out = torch.empty_like(x)
    n = x.numel()
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(n, BLOCK_SIZE),)
    relu_kernel[grid](x, out, n, BLOCK_SIZE=BLOCK_SIZE)
    return out

In [ ]:
x = torch.randn(2 ** 20, device='cuda', dtype=torch.float32)
out = relu(x)
assert torch.allclose(out, x.clamp(min=0))

---
### Mean (Reduction)

Computes $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$ â€” the average value across all elements.

Fast because each program independently sums its block in parallel (O(n) â†’ O(n/BLOCK_SIZE) parallel steps), with a cheap final CPU-side combine.

Cell 1: GPU kernel to compute partial sums per tile

Cell 2: CPU wrapper to combine partial sums and divide

Cell 3: Usage demo

In [ ]:
@triton.jit
def partial_sum_kernel(x_ptr, partial_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0) 
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offs < n                                   
    x = tl.load(x_ptr + offs, mask=mask, other=0.0) 
    tl.store(partial_ptr + pid, tl.sum(x, axis=0))

In [ ]:
def mean(x: torch.Tensor) -> torch.Tensor:
    assert x.is_cuda
    n = x.numel()
    BLOCK_SIZE = 1024
    n_programs = triton.cdiv(n, BLOCK_SIZE)
    partial = torch.empty(n_programs, device='cuda', dtype=torch.float32) # one slot per partial sum
    grid = (n_programs,)
    partial_sum_kernel[grid](x, partial, n, BLOCK_SIZE=BLOCK_SIZE)
    return partial.sum() / n # combine partial sums and divide. Maybe recursive approach warranted if gigantic vector but not likely useful

In [ ]:
x = torch.randn(2 ** 20, device='cuda', dtype=torch.float32)
out = mean(x)
assert torch.allclose(out, x.mean(), atol=1e-4)

---
### GeMM (General Matrix Multiply)

Computes $C = A \times B$ where $A \in \mathbb{R}^{M \times K}$, $B \in \mathbb{R}^{K \times N}$, $C \in \mathbb{R}^{M \times N}$ â€” the core operation behind all neural network layers.

Fast because `tl.dot` maps directly to GPU tensor cores (specialized hardware for matrix multiply), and tiles are accumulated entirely in registers without round-tripping to DRAM mid-computation.

Cell 1: GPU kernel â€” each program computes one [BM, BN] output tile by stepping through K

Cell 2: CPU wrapper

Cell 3: Usage demo

In [ ]:
@triton.jit
def matmul_kernel(a_ptr, b_ptr, c_ptr, M, N, K,
                  stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
                  BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pid_m = tl.program_id(0) # which row-tile of C this program computes
    pid_n = tl.program_id(1) # which col-tile of C this program computes

    offs_m = pid_m * BM + tl.arange(0, BM) # row indices for this tile in A and C
    offs_n = pid_n * BN + tl.arange(0, BN) # col indices for this tile in B and C
    offs_k = tl.arange(0, BK) # indices along the shared K dimension (advances each loop iteration)

    # build 2D pointer blocks via broadcasting: [:, None] makes a column vector, [None, :] makes a row vector
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak # [BM, BK] pointer block into A
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn # [BK, BN] pointer block into B

    acc = tl.zeros((BM, BN), dtype=tl.float32) # accumulator for output tile, starts at zero
    for k in range(tl.cdiv(K, BK)): # step through K dimension one BK-wide slice at a time
        a = tl.load(a_ptrs, mask=(offs_m[:, None] < M) & (offs_k[None, :] < K - k*BK), other=0.0) # load A tile; mask boundary
        b = tl.load(b_ptrs, mask=(offs_k[:, None] < K - k*BK) & (offs_n[None, :] < N), other=0.0) # load B tile; mask boundary
        acc = tl.dot(a, b, acc) # fused multiply-accumulate: acc += a @ b, uses tensor cores
        a_ptrs += BK * stride_ak # advance A pointer to next K slice
        b_ptrs += BK * stride_bk # advance B pointer to next K slice

    offs_cm = pid_m * BM + tl.arange(0, BM) # row indices for writing output tile to C
    offs_cn = pid_n * BN + tl.arange(0, BN) # col indices for writing output tile to C
    c_ptrs = c_ptr + offs_cm[:, None] * stride_cm + offs_cn[None, :] * stride_cn #2D pointer block into C
    tl.store(c_ptrs, acc.to(tl.float16), mask=(offs_cm[:, None] < M) & (offs_cn[None, :] < N)) # write; cast to fp16; mask boundary

In [ ]:
def matmul(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    assert a.is_cuda and b.is_cuda
    assert a.shape[1] == b.shape[0], "incompatible dimensions"
    M, K = a.shape
    K, N = b.shape
    a = a.to(torch.float16) # tensor cores require fp16 input
    b = b.to(torch.float16)
    c = torch.empty((M, N), device='cuda', dtype=torch.float16)
    BM, BN, BK = 64, 64, 32 # tile sizes; tune for your GPU
    grid = (triton.cdiv(M, BM), triton.cdiv(N, BN)) # one program per output tile
    matmul_kernel[grid](
        a, b, c, M, N, K,
        a.stride(0), a.stride(1), # strides tell the kernel how many elements to skip per row/col step
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BM=BM, BN=BN, BK=BK
    )
    return c

In [ ]:
M, K, N = 512, 512, 512
a = torch.randn(M, K, device='cuda')
b = torch.randn(K, N, device='cuda')
c = matmul(a, b)
ref = (a.float() @ b.float()).half()
assert torch.allclose(c, ref, atol=1e-1) # fp16 accumulation has larger tolerance

---
### Fused Softmax

Softmax converts a row of logits into probabilities: $\text{softmax}(x)_i = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$.

Fast because naive softmax requires 3 DRAM passes (max scan â†’ exp+sum â†’ divide); fused softmax keeps the entire row in registers and does all three in one pass, cutting memory traffic by ~3Ã—.

Cell 1: GPU kernel â€” one program per row, entire row fits in registers

Cell 2: CPU wrapper

Cell 3: Usage demo

In [ ]:
@triton.jit
def softmax_kernel(out_ptr, x_ptr, row_stride, n_rows, n_cols, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0) # one program per row
    offs = tl.arange(0, BLOCK_SIZE) # column indices (BLOCK_SIZE must fit the entire row)
    mask = offs < n_cols # guard against padding columns beyond actual row length

    x = tl.load(x_ptr + row * row_stride + offs, mask=mask, other=-float('inf')) # load entire row; OOB slots get -inf so they don't affect max/sum
    x = x - tl.max(x, axis=0) # subtract row max for numerical stability (prevents exp overflow)
    x = tl.exp(x) # exponentiate shifted values
    x = x / tl.sum(x, axis=0) # normalize by sum to get probabilities

    tl.store(out_ptr + row * row_stride + offs, x, mask=mask) # write softmax output back to GPU memory

In [ ]:
def softmax(x: torch.Tensor) -> torch.Tensor:
    assert x.is_cuda and x.ndim == 2
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols) # must be power of 2 and large enough to hold the full row
    out = torch.empty_like(x)
    grid = (n_rows,) # one program per row
    softmax_kernel[grid](out, x, x.stride(0), n_rows, n_cols, BLOCK_SIZE=BLOCK_SIZE)
    return out

In [ ]:
x = torch.randn(128, 512, device='cuda', dtype=torch.float32)
out = softmax(x)
assert torch.allclose(out, torch.softmax(x, dim=1), atol=1e-4)

---
### Batch Norm

Normalizes each feature column across the batch: $\hat{x} = \frac{x - \mu}{\sigma} \cdot \gamma + \beta$, where $\mu$ and $\sigma$ are the column mean and std.

Fast because each of the C features gets its own independent program â€” all C normalizations run simultaneously instead of sequentially.

Cell 1: GPU kernel â€” one program per feature column, three passes (mean â†’ variance â†’ normalize)

Cell 2: CPU wrapper

Cell 3: Usage demo

In [ ]:
@triton.jit
def batchnorm_kernel(X, Y, W, B, N, C, eps, BLOCK_SIZE: tl.constexpr):
    feat = tl.program_id(0) # one program per feature (column)

    # Pass 1: mean
    _sum = tl.zeros([BLOCK_SIZE], dtype=tl.float32) # accumulator for partial sums across rows
    for off in range(0, N, BLOCK_SIZE): # iterate over rows in BLOCK_SIZE chunks
        rows = off + tl.arange(0, BLOCK_SIZE) # row indices for this chunk
        x = tl.load(X + rows * C + feat, mask=rows < N, other=0.0).to(tl.float32) # load this feature's values for these rows
        _sum += x # accumulate partial sum
    mean = tl.sum(_sum, axis=0) / N # reduce partial sums to scalar mean

    # Pass 2: variance
    _var = tl.zeros([BLOCK_SIZE], dtype=tl.float32) # accumulator for partial variance
    for off in range(0, N, BLOCK_SIZE):
        rows = off + tl.arange(0, BLOCK_SIZE)
        x = tl.load(X + rows * C + feat, mask=rows < N, other=0.0).to(tl.float32) # reload feature values
        d = tl.where(rows < N, x - mean, 0.0) # deviation from mean; zero out OOB slots
        _var += d * d # accumulate squared deviations
    rstd = 1.0 / tl.sqrt(tl.sum(_var, axis=0) / N + eps) # reciprocal std dev; eps prevents division by zero

    # Pass 3: normalize
    w = tl.load(W + feat) # learnable scale parameter for this feature
    b_val = tl.load(B + feat) # learnable bias parameter for this feature
    for off in range(0, N, BLOCK_SIZE):
        rows = off + tl.arange(0, BLOCK_SIZE)
        mask = rows < N
        x = tl.load(X + rows * C + feat, mask=mask, other=0.0).to(tl.float32) # reload feature values
        tl.store(Y + rows * C + feat, (x - mean) * rstd * w + b_val, mask=mask) # normalize, scale, shift, write output

In [ ]:
def batch_norm(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    assert x.is_cuda and x.ndim == 2 # x shape: [N, C]
    N, C = x.shape
    out = torch.empty_like(x)
    grid = (C,) # one program per feature column
    batchnorm_kernel[grid](x, out, w, b, N, C, eps, BLOCK_SIZE=min(1024, triton.next_power_of_2(N)))
    return out

In [ ]:
N, C = 512, 64
x = torch.randn(N, C, device='cuda', dtype=torch.float32)
w = torch.ones(C, device='cuda', dtype=torch.float32) # scale initialized to 1
b = torch.zeros(C, device='cuda', dtype=torch.float32) # bias initialized to 0
out = batch_norm(x, w, b)
ref = torch.nn.functional.batch_norm(x, None, None, w, b, training=True)
assert torch.allclose(out, ref, atol=1e-4)

---
### Max Pooling

Slides a window of size `kernel_size` across the input with a given `stride`, outputting the maximum value in each window: $y_i = \max(x_{i \cdot s}, \ldots, x_{i \cdot s + k - 1})$.

Fast because all output elements are independent of each other â€” every program computes its window maximum simultaneously.

Cell 1: GPU kernel â€” one program per output element, reduces its window with `tl.max`

Cell 2: CPU wrapper

Cell 3: Usage demo

In [ ]:
@triton.jit
def max_pool1d_kernel(X_ptr, Out_ptr, L, kernel_size, stride, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0) # which output element this program computes
    in_start = pid * stride # where this output's receptive field starts in the input
    offs = in_start + tl.arange(0, BLOCK_SIZE) # indices of input elements in the kernel window
    mask = (offs < L) & (offs < in_start + kernel_size) # stay within input bounds AND within kernel window
    x = tl.load(X_ptr + offs, mask=mask, other=-float('inf')) # load window; OOB slots get -inf so they don't win the max
    tl.store(Out_ptr + pid, tl.max(x, axis=0)) # reduce window to scalar max, write to output

In [ ]:
def max_pool1d(x: torch.Tensor, kernel_size: int, stride: int) -> torch.Tensor:
    assert x.is_cuda and x.ndim == 1
    L = x.numel()
    L_out = (L - kernel_size) // stride + 1 # number of output elements
    out = torch.empty(L_out, device='cuda', dtype=x.dtype)
    BLOCK_SIZE = triton.next_power_of_2(kernel_size) # must be power of 2 and >= kernel_size
    grid = (L_out,) # one program per output element
    max_pool1d_kernel[grid](x, out, L, kernel_size, stride, BLOCK_SIZE=BLOCK_SIZE)
    return out

In [ ]:
x = torch.randn(1024, device='cuda', dtype=torch.float32)
out = max_pool1d(x, kernel_size=4, stride=2)
ref = torch.nn.functional.max_pool1d(x.unsqueeze(0).unsqueeze(0), 4, stride=2).squeeze()
assert torch.allclose(out, ref)

---
### Global Average Pooling

Collapses the spatial dimensions $H \times W$ of a feature map to a single value per channel: $y_{b,c} = \frac{1}{HW}\sum_{h,w} x_{b,c,h,w}$.

Cell 1: GPU kernel â€” one program per (batch, channel) pair, sums all spatial elements and divides

Cell 2: CPU wrapper

Cell 3: Usage demo

In [ ]:
@triton.jit
def gap_kernel(X_ptr, Out_ptr, spatial, BLOCK_SIZE: tl.constexpr):
    bc = tl.program_id(0) # flattened (batch, channel) index â€” one program per (batch, channel) pair
    base = bc * spatial # offset to the start of this (batch, channel)'s spatial data in the flat array
    _acc = tl.zeros([BLOCK_SIZE], dtype=tl.float32) # accumulator for partial sums over spatial elements
    for off in range(0, spatial, BLOCK_SIZE): # iterate over spatial elements in chunks
        offs = off + tl.arange(0, BLOCK_SIZE) # indices within the spatial chunk
        _acc += tl.load(X_ptr + base + offs, mask=offs < spatial, other=0.0).to(tl.float32) # accumulate
    tl.store(Out_ptr + bc, tl.sum(_acc, axis=0) / spatial) # reduce to scalar mean, write to output

In [ ]:
def global_average_pool(x: torch.Tensor) -> torch.Tensor:
    assert x.is_cuda and x.ndim == 4 # x shape: [B, C, H, W]
    B, C, H, W = x.shape
    spatial = H * W
    x_flat = x.reshape(B * C, spatial) # flatten to [B*C, H*W] for 1D indexing
    out = torch.empty(B * C, device='cuda', dtype=torch.float32)
    BLOCK_SIZE = min(1024, triton.next_power_of_2(spatial))
    grid = (B * C,) # one program per (batch, channel) pair
    gap_kernel[grid](x_flat, out, spatial, BLOCK_SIZE=BLOCK_SIZE)
    return out.reshape(B, C) # return shape [B, C]

In [ ]:
x = torch.randn(8, 16, 14, 14, device='cuda', dtype=torch.float32) # batch=8, channels=16, 14x14 spatial
out = global_average_pool(x) # shape: [8, 16]
ref = x.mean(dim=[2, 3])
assert torch.allclose(out, ref, atol=1e-4)